# Step 1 - Load data and first look

Going to load all 5 years and see what we're working with before doing anything else.

In [ ]:
import pandas as pd
import numpy as np
import os

# data is in ../data/ symlinked or copied from Downloads
DATA_DIR = '../data/'

In [ ]:
# load one year first to check it works and see the shape
# using sep='|' because it's pipe-separated
# low_memory=False because mixed types will cause warnings otherwise

df_2021 = pd.read_csv(
    DATA_DIR + 'ValeursFoncieres-2021.txt',
    sep='|',
    low_memory=False
)

print(df_2021.shape)
df_2021.head(3)

In [ ]:
# ok so how many rows per year roughly?
# going to load them all and add a 'year' column to keep track

years = [2021, 2022, 2023, 2024, 2025]
dfs = []

for year in years:
    path = DATA_DIR + f'ValeursFoncieres-{year}.txt'
    df = pd.read_csv(path, sep='|', low_memory=False)
    df['annee'] = year  # add year column
    dfs.append(df)
    print(f'{year}: {df.shape[0]:,} rows')

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal: {df.shape[0]:,} rows, {df.shape[1]} columns')

In [ ]:
# sanity check - do all years have the same columns?
# had issues before where different years had different column counts
for i, d in enumerate(dfs):
    print(years[i], list(d.columns))

In [ ]:
df.dtypes

In [ ]:
# check missing values - expecting a lot given what the data looks like
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
summary[summary.missing_count > 0].sort_values('missing_pct', ascending=False)

In [ ]:
# at first I thought the lot columns (1er lot, 2eme lot...) would be useful
# but they're probably mostly empty for simple house sales
# let's check
lot_cols = [c for c in df.columns if 'lot' in c.lower()]
print(lot_cols)
df[lot_cols].isnull().mean().round(3)

In [ ]:
# the main target column is 'Valeur fonciere' (property value)
# it's stored as string with comma decimal separator (French format)
# need to fix this

print(df['Valeur fonciere'].dtype)
print(df['Valeur fonciere'].head(10))

In [ ]:
# convert to float
df['valeur_fonciere'] = (
    df['Valeur fonciere']
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# quick sanity check
print(df['valeur_fonciere'].describe())

In [ ]:
# that max value looks suspicious - let's see
# also 0 values probably mean something went wrong or it's a gift/transfer
print('Zeros:', (df['valeur_fonciere'] == 0).sum())
print('Very high (>10M):', (df['valeur_fonciere'] > 10_000_000).sum())
df[df['valeur_fonciere'] > 10_000_000][['Commune', 'Nature mutation', 'valeur_fonciere', 'annee']].head(10)

In [ ]:
# what types of transactions are there?
df['Nature mutation'].value_counts()

In [ ]:
# what types of property?
df['Type local'].value_counts(dropna=False)

In [ ]:
# save merged df for next notebook
# TODO: might be better to save as parquet for speed, try that later
df.to_csv('../data/merged_raw.csv', index=False)
print('saved')